**Importer les libraries pertinentes**

Source : Prettenhofer, P., Grisel, O., Blondel, M., Amor, A. et Buitinck, L. [Classification of text documents using sparse features](https://scikit-learn.org/stable/auto_examples/text/plot_document_classification_20newsgroups.html#sphx-glr-auto-examples-text-plot-document-classification-20newsgroups-py). *scikit-learn*. 

In [2]:
import pandas as pd
from os import listdir
from os import path

from IPython.display import clear_output

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Naive Bayes
from sklearn.naive_bayes import MultinomialNB

# K plus proches voisins (kNN)
from sklearn.neighbors import KNeighborsClassifier

# Machines à vecteurs de support (SVM)
from sklearn import svm

# Régression Logistique
from sklearn.linear_model import LogisticRegression

# Forêt aléatoire 
from sklearn.ensemble import RandomForestClassifier

# Perceptron multicouche
from sklearn.neural_network import MLPClassifier

**Charger les données**

In [3]:
folder = '../1-data/training_datasets/'
datasets = listdir(folder)
datasets = datasets[5:6]
datasets

['train_dataset_60pc.xlsx']

**Entraîner les modèles et sortir les résultats**

In [4]:
report = []
for file in datasets:
    ratio = file[14:-7]
    
    df = pd.read_excel(path.join(folder, file))
    X = df['text_post'].astype(str)
    y = df['category']

    # 5-fold cross-validation
    stratified_kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    # Valeurs à tester pour le nombre de traits discriminants
    n_features_values = [100, 250, 500, 1000, 2500, 5000, 10000, 15000]

    # Antidictionnaire personnalisé
    with open('./utils/exclusion_v2.stop', encoding='utf-8') as f:
        custom_stopwords = [x.lower().strip('\n') for x in f.readlines()]
        custom_stopwords += list(ENGLISH_STOP_WORDS)

    # Algorithmes à tester
    algorithms = {
        'Naive Bayes' : MultinomialNB(), 
        'kNN (k=1)' : KNeighborsClassifier(n_neighbors=1),
        'kNN (k=2)' : KNeighborsClassifier(n_neighbors=2),
        'kNN (k=3)' : KNeighborsClassifier(n_neighbors=3),
        'Support Vector Machines (SVM)' :  svm.SVC(kernel="linear", C=1.0),
        'Logistic Regression': LogisticRegression(C=5, max_iter=1000),
        'Random Forest': RandomForestClassifier(),
        #'Multilayered Perceptron': MLPClassifier(hidden_layer_sizes=(100,), max_iter=500)
    }

    for n_features in n_features_values:
            tfidf_bow_vectorizer = TfidfVectorizer(stop_words=custom_stopwords, max_features=n_features)
            
            for algorithm in algorithms.keys():
                pipeline_algo = Pipeline([
                    ('tfidf', tfidf_bow_vectorizer),
                    ('classifier', algorithms[algorithm])
                ])

                # Validation croisée 
                scores = cross_val_score(pipeline_algo, X, y, cv=stratified_kfold, scoring='accuracy')
                predictions = cross_val_predict(pipeline_algo, X, y, cv=stratified_kfold)

                # Performances (rappel, précision, score F)
                results = {
                    '% Incels' : int(ratio),
                    'Algorithme' : algorithm,
                    'N traits discr.' : n_features,
                    'accuracy': scores.mean()       
                }
                results.update(classification_report(y, predictions, output_dict=True)['macro avg'])
                report.append(results)
                
                print(algorithm, f' | {ratio}% Incels | ', f'{n_features} traits discr.\n', classification_report(y, predictions))
                clear_output(wait=True)

KeyboardInterrupt: 

**Résultats**

In [5]:
# N.B. : Support = Nombre d'occurrence dans chaque classe
report = pd.DataFrame(report)
report['nb_posts_incels'] = (report['% Incels'].apply(lambda x: x/100) * report['support']).astype(int)
report['nb_posts_neutral'] = (report['% Incels'].apply(lambda x: 1-(x/100)) * report['support']).astype(int)

report = report.rename(columns={'support':'nb_posts_total'})
report['nb_posts_total'] = report['nb_posts_total'].astype(int)

# Réorganiser l'ordre des colonnes
report = report[['nb_posts_total', 'nb_posts_incels', 'nb_posts_neutral', '% Incels', 'Algorithme', 
                 'N traits discr.', 'accuracy', 'precision', 'recall','f1-score']]

report

,nb_posts_total,nb_posts_incels,nb_posts_neutral,% Incels,Algorithme,N traits discr.,accuracy,precision,recall,f1-score
0,50000,30000,20000,60,Naive Bayes,100,0.61638,0.590342,0.549567,0.522196
1,50000,30000,20000,60,kNN (k=1),100,0.58572,0.506639,0.502067,0.433247
2,50000,30000,20000,60,kNN (k=2),100,0.59878,0.542048,0.505542,0.408379
3,50000,30000,20000,60,kNN (k=3),100,0.59210,0.523690,0.506475,0.433247
4,50000,30000,20000,60,Support Vector Machines (SVM),100,0.63850,0.647698,0.652817,0.637238
5,50000,30000,20000,60,Logistic Regression,100,0.64134,0.642540,0.648458,0.638136
6,50000,30000,20000,60,Random Forest,100,0.63284,0.635856,0.641367,0.629864
7,50000,30000,20000,60,Naive Bayes,250,0.64122,0.629964,0.581217,0.566024
8,50000,30000,20000,60,kNN (k=1),250,0.58140,0.514370,0.506517,0.458957
9,50000,30000,20000,60,kNN (k=2),250,0.60116,0.559270,0.510217,0.421986


**Exporter les résultats de l'apprentissage**

In [6]:
report.sort_values(by='f1-score', ascending=False).to_csv(
    f'../3-results/results_training_60pc_10000-15000_traits.csv', index=False
)

**Tester le système retenu sur de nouvelles données**

*On reproduit l'apprentissage pour le système retenu uniquement*

In [ ]:
file_train = '../1-data/training_datasets/train_dataset_40pc.xlsx'

df_train = pd.read_excel(file_train)
df_train

In [ ]:
# Système retenu : Naive Bayes ; 40% de données Incels ; 5 000 traits discriminants
X_train = df_train['text_post'].astype(str)
y_train = df_train['category']

ngram_values = (1, 2)

# Antidictionnaire personnalisé
with open('./utils/exclusion_v2.stop', encoding='utf-8') as f:
    custom_stopwords = [x.lower().strip('\n') for x in f.readlines()]
    custom_stopwords += list(ENGLISH_STOP_WORDS)

pipeline_nb = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words=custom_stopwords, ngram_range=ngram_values, max_features=15000)),
    ('nb', MultinomialNB())
])

pipeline_nb.fit(X_train, y_train)

In [ ]:
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
nb_predictions = cross_val_predict(pipeline_nb, X_train, y_train, cv=stratified_kfold)

# Performances en phase d'apprentissage du modèle retenu (rappel, précision, score F)
# Calculer et afficher les performances
accuracy_nb = accuracy_score(y_train, nb_predictions)
precision_nb = precision_score(y_train, nb_predictions, average='macro')
recall_nb = recall_score(y_train, nb_predictions, average='macro')
f1_nb = f1_score(y_train, nb_predictions, average='macro')

df = [
    {
        'Phase': 'Apprentissage',
        'Précision': precision_nb,
        'Rappel': recall_nb,
        'Mesure F': f1_nb,
        'Exactitude': accuracy_nb

    }
]

print(classification_report(y_train, nb_predictions))

*On teste sur de nouvelles données*

In [ ]:
file_test = '../1-data/test_dataset_10pc.xlsx'

df_test = pd.read_excel(file_test)
df_test

*Vérifier qu'aucune donnée test n'était contenue dans les données d'appentissage*

In [ ]:
validate = pd.concat([df_train, df_test])

validate[validate.duplicated(subset='id_post')]

*Appliquer le classifieur aux nouvelles données*

In [ ]:
X_test = df_test['text_post'].astype(str)
y_test = df_test['category']

y_pred_nb = pipeline_nb.predict(X_test)

In [ ]:
# Calculer et afficher les performances
accuracy_nb = accuracy_score(y_test, y_pred_nb)
precision_nb = precision_score(y_test, y_pred_nb, average='macro')
recall_nb = recall_score(y_test, y_pred_nb, average='macro')
f1_nb = f1_score(y_test, y_pred_nb, average='macro')

df.append(
    {
        'Phase': 'Test',
        'Précision': precision_nb,
        'Rappel': recall_nb,
        'Mesure F': f1_nb,
        'Exactitude': accuracy_nb

    }
)

pd.set_option("display.precision", 4)
df = pd.DataFrame(df)
df.to_csv('../3-results/results_test.csv', index=False)
df

In [ ]:
accuracy_nb

In [ ]:
precision_nb

In [ ]:
recall_nb

In [ ]:
f1_nb

In [ ]:
print(classification_report(y_test, y_pred_nb))

In [ ]:
import pandas as pd
results = pd.read_csv('../3-results/results_training.csv')
results.sort_values(by='f_score', ascending=False)